# Emotion Vector Analysis

This notebook is the post-training analysis stage for the project.

It can use either:
- the checkpoint and metadata already present in the current runtime
- or the Google Drive backup created by `02_wav2vec_finetuning.ipynb`

Important note:
- the checkpoint backup alone is not enough for embedding extraction
- this notebook still needs access to the local `RAVDESS` audio files referenced by the metadata CSV
- so the smoothest path is still to run this notebook in the same Colab runtime right after notebook 02, or at least in a runtime where the dataset has already been prepared under `/content/speech-emotion-directions/data/raw`

What it does:
- loads the trained `wav2vec2` checkpoint
- extracts pooled hidden representations from every layer
- checks how separable each layer is with a nearest-centroid classifier
- builds emotion directions relative to `neutral`
- adds nuisance-controlled analysis by centering within actor and statement groups
- visualizes the final-layer latent space
- runs a small steering-style intervention on the classifier input space
- optionally syncs the analysis outputs back to Google Drive

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)


def run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)



def ensure_packages() -> None:
    package_map = {
        "yaml": "pyyaml",
        "pandas": "pandas",
        "numpy": "numpy",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "umap": "umap-learn",
        "torch": "torch",
        "transformers": "transformers",
        "datasets": "datasets",
        "accelerate": "accelerate",
        "sklearn": "scikit-learn",
        "tqdm": "tqdm",
        "huggingface_hub": "huggingface_hub",
    }
    missing = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing:
        run_command([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Required analysis packages are already available.")



def find_local_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd / "FinalProject"]:
        candidate = candidate.resolve()
        if (candidate / "configs" / "wav2vec.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root locally.")



def repo_status_lines(repo_root: Path) -> list[str]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--short"],
        text=True,
        capture_output=True,
        check=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]



def repo_ahead_behind(repo_root: Path) -> tuple[int, int]:
    result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-list", "--left-right", "--count", "HEAD...origin/main"],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0 or not result.stdout.strip():
        return 0, 0
    ahead_str, behind_str = result.stdout.strip().split()
    return int(ahead_str), int(behind_str)



def clone_clean_code_checkout(clean_root: Path) -> Path:
    if clean_root.exists():
        shutil.rmtree(clean_root)
    run_command(["git", "clone", "--depth", "1", REPO_URL, str(clean_root)])
    return clean_root



def prepare_project_and_code_roots() -> tuple[Path, Path]:
    runtime_root = Path("/content") / REPO_NAME
    clean_root = Path("/content") / f"{REPO_NAME}-clean"

    if runtime_root.exists() and (runtime_root / ".git").exists():
        try:
            run_command(["git", "-C", str(runtime_root), "fetch", "origin"])
        except subprocess.CalledProcessError as exc:
            print(f"Fetch failed for existing Colab repo; continuing with local state. Details: {exc}")

        status_lines = repo_status_lines(runtime_root)
        ahead, behind = repo_ahead_behind(runtime_root)

        if not status_lines and ahead == 0:
            if behind > 0:
                try:
                    run_command(["git", "-C", str(runtime_root), "pull", "--ff-only", "origin", "main"])
                    code_root = runtime_root
                except subprocess.CalledProcessError as exc:
                    print(f"Fast-forward pull failed; using a clean code checkout instead. Details: {exc}")
                    code_root = clone_clean_code_checkout(clean_root)
            else:
                code_root = runtime_root
        else:
            print(f"Using a clean code checkout because {runtime_root} has local changes or local commits.")
            for line in status_lines[:10]:
                print(" -", line)
            if ahead or behind:
                print(f"Repo divergence relative to origin/main: ahead={ahead}, behind={behind}")
            code_root = clone_clean_code_checkout(clean_root)

        project_root = runtime_root
    elif runtime_root.exists():
        print(f"Using existing non-git project directory for data/artifacts: {runtime_root}")
        project_root = runtime_root
        code_root = clone_clean_code_checkout(clean_root)
    else:
        run_command(["git", "clone", "--depth", "1", REPO_URL, str(runtime_root)])
        project_root = runtime_root
        code_root = runtime_root

    return project_root, code_root


ensure_packages()

if IS_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT, CODE_ROOT = prepare_project_and_code_roots()
else:
    PROJECT_ROOT = find_local_project_root()
    CODE_ROOT = PROJECT_ROOT

def module_uses_code_root(module_file: str | None, code_root: Path) -> bool:
    if module_file is None:
        return False
    try:
        Path(module_file).resolve().relative_to(code_root.resolve())
        return True
    except ValueError:
        return False


os.chdir(PROJECT_ROOT)
for root in [str(CODE_ROOT), str(PROJECT_ROOT)]:
    while root in sys.path:
        sys.path.remove(root)

sys.path.insert(0, str(CODE_ROOT))
if str(PROJECT_ROOT) != str(CODE_ROOT):
    sys.path.insert(1, str(PROJECT_ROOT))

src_module = sys.modules.get("src")
src_file = getattr(src_module, "__file__", None)
if src_module is not None and not module_uses_code_root(src_file, CODE_ROOT):
    stale_modules = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
    for name in stale_modules:
        del sys.modules[name]

print("Project root:", PROJECT_ROOT)
print("Code root:", CODE_ROOT)
print("Working directory:", Path.cwd().resolve())

In [ ]:
import json
from pathlib import Path

from src.data.dataset import load_project_metadata

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_metadata_path = PROJECT_ROOT / 'data' / 'metadata' / 'ravdess_metadata.csv'
local_raw_audio_root = PROJECT_ROOT / 'data' / 'raw' / 'ravdess_audio_speech'

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root is not None else None
drive_metadata_path = drive_run_root / 'metadata' / 'ravdess_metadata.csv' if drive_run_root is not None else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root is not None else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir is not None and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir

metadata_path = local_metadata_path
if not metadata_path.exists() and drive_metadata_path is not None and drive_metadata_path.exists():
    metadata_path = drive_metadata_path
original_metadata_path = metadata_path

analysis_dir = local_analysis_dir
analysis_dir.mkdir(parents=True, exist_ok=True)
resolved_metadata_path = analysis_dir / 'ravdess_metadata_resolved.csv'

required_checkpoint_files = [
    checkpoint_dir / 'model_state.pt',
    checkpoint_dir / 'config.json',
    checkpoint_dir / 'label_mapping.json',
]
missing_checkpoint = [str(path) for path in required_checkpoint_files if not path.exists()]
assert not missing_checkpoint, (
    'Missing required checkpoint files for analysis. Checked: ' + ', '.join(missing_checkpoint)
)
assert original_metadata_path.exists(), f'Metadata CSV not found: {original_metadata_path}'
assert local_raw_audio_root.exists(), (
    'Local RAVDESS audio directory is missing. Notebook 03 still needs the audio files to extract embeddings. '
    f'Expected: {local_raw_audio_root}. Run notebook 02 data prep in this runtime first, or restore the dataset under that path.'
)

resolved_metadata = load_project_metadata(
    metadata_path=original_metadata_path,
    raw_audio_root=local_raw_audio_root,
)

if IS_COLAB:
    unresolved_user_paths = resolved_metadata['file_path'].astype(str).str.startswith('/Users/')
    assert not unresolved_user_paths.any(), (
        'Resolved metadata still contains local /Users paths in Colab. '
        'This notebook should only continue with Colab-accessible audio paths.'
    )

resolved_metadata.to_csv(resolved_metadata_path, index=False)
metadata_path = resolved_metadata_path
metadata_preview = resolved_metadata.head(3)
path_preview = metadata_preview['file_path'].tolist() if 'file_path' in metadata_preview.columns else []

print('Checkpoint source:', checkpoint_dir)
print('Original metadata source:', original_metadata_path)
print('Resolved metadata source:', resolved_metadata_path)
print('Active metadata path for downstream cells:', metadata_path)
print('Analysis output dir:', analysis_dir)
if drive_run_root is not None:
    print('Drive run root:', drive_run_root)
print('Local raw audio root:', local_raw_audio_root)
print('Sample resolved file_path values from metadata:')
for value in path_preview:
    print(' -', value)

print('\nCheckpoint files:')
for path in sorted(checkpoint_dir.iterdir()):
    print(' -', path.name)


In [ ]:
from src.analysis.extract_embeddings import extract_and_save_embeddings

analysis_output_dir = extract_and_save_embeddings(
    checkpoint_dir=checkpoint_dir,
    metadata_path=metadata_path,
    output_dir=analysis_dir,
    batch_size=32,
    num_workers=0,
    raw_audio_root=local_raw_audio_root,
)
print('Saved embedding artifacts to:', analysis_output_dir)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.analysis.emotion_vectors import evaluate_layerwise_centroid_classifier, load_embedding_artifacts

artifacts = load_embedding_artifacts(analysis_dir)
label_names = list(artifacts.summary["label_names"])
layer_metrics = evaluate_layerwise_centroid_classifier(
    layer_embeddings=artifacts.layer_embeddings,
    split_names=artifacts.metadata["split"],
    label_ids=artifacts.true_label_ids,
    label_names=label_names,
)
layer_metrics.to_csv(analysis_dir / "layerwise_centroid_metrics.csv", index=False)

val_metrics = layer_metrics[layer_metrics["split"] == "val"].sort_values("macro_f1", ascending=False)
best_layer_idx = int(val_metrics.iloc[0]["layer_index"])
final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)

print("Best validation layer by nearest-centroid macro F1:", best_layer_idx)
print("Final classifier layer:", final_layer_idx)
display(layer_metrics.head())
display(val_metrics.head())

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=layer_metrics[layer_metrics["split"].isin(["val", "test"])],
    x="layer_index",
    y="macro_f1",
    hue="split",
    marker="o",
)
plt.title("Layer-wise Nearest-Centroid Macro F1")
plt.xlabel("Layer Index")
plt.ylabel("Macro F1")
plt.tight_layout()
plt.savefig(analysis_dir / "layerwise_macro_f1.png", dpi=200)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from src.analysis.emotion_vectors import (
    build_direction_vectors,
    center_within_groups,
    compute_class_centroids,
    pairwise_cosine_matrix,
    summarize_projection_means,
)

reference_label = "neutral"
reference_idx = label_names.index(reference_label)
split_array = artifacts.metadata["split"].to_numpy()
train_mask = split_array == "train"
test_mask = split_array == "test"

final_layer_embeddings = artifacts.layer_embeddings[:, final_layer_idx]
actor_centered = center_within_groups(final_layer_embeddings, artifacts.metadata["actor_id"].astype(str))
actor_statement_centered = center_within_groups(actor_centered, artifacts.metadata["statement_code"].astype(str))

train_embeddings = actor_statement_centered[train_mask]
train_label_ids = artifacts.true_label_ids[train_mask]
test_embeddings = actor_statement_centered[test_mask]
test_label_ids = artifacts.true_label_ids[test_mask]

centroids = compute_class_centroids(train_embeddings, train_label_ids, num_labels=len(label_names))
directions = build_direction_vectors(centroids, reference_idx)
centroid_cosine_df = pd.DataFrame(
    pairwise_cosine_matrix(centroids),
    index=label_names,
    columns=label_names,
)
direction_names = [f"{name}_minus_neutral" for name in label_names]
projection_summary_df = summarize_projection_means(
    embeddings=test_embeddings,
    label_ids=test_label_ids,
    directions=directions,
    label_names=label_names,
    direction_names=direction_names,
)

centroid_cosine_df.to_csv(analysis_dir / "centroid_cosine_matrix.csv")
projection_summary_df.to_csv(analysis_dir / "projection_summary_test_centered.csv", index=False)

print("Centroid cosine similarity matrix (train, actor+statement centered):")
display(centroid_cosine_df.round(3))
print("Mean test-set projection summary (actor+statement centered):")
display(projection_summary_df.round(3))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import umap

reducer = umap.UMAP(random_state=42)
test_umap = reducer.fit_transform(test_embeddings)
plot_df = pd.DataFrame(
    {
        "umap_1": test_umap[:, 0],
        "umap_2": test_umap[:, 1],
        "label": [label_names[idx] for idx in test_label_ids],
        "actor_id": artifacts.metadata.loc[test_mask, "actor_id"].astype(str).to_list(),
    }
)
plot_df.to_csv(analysis_dir / "test_umap_coordinates.csv", index=False)

plt.figure(figsize=(9, 7))
sns.scatterplot(data=plot_df, x="umap_1", y="umap_2", hue="label", alpha=0.85, s=55)
plt.title("UMAP of Test Embeddings (Actor+Statement Centered, Final Layer)")
plt.tight_layout()
plt.savefig(analysis_dir / "test_umap.png", dpi=220)
plt.show()

In [ ]:
from IPython.display import display

from src.analysis.emotion_vectors import evaluate_direction_steering
from src.analysis.extract_embeddings import load_trained_checkpoint

checkpoint = load_trained_checkpoint(checkpoint_dir)
classifier_weight = checkpoint.model.classifier.weight.detach().cpu().numpy()
classifier_bias = checkpoint.model.classifier.bias.detach().cpu().numpy()

raw_train_embeddings = final_layer_embeddings[train_mask]
raw_test_embeddings = final_layer_embeddings[test_mask]
raw_centroids = compute_class_centroids(raw_train_embeddings, train_label_ids, num_labels=len(label_names))
raw_directions = build_direction_vectors(raw_centroids, reference_idx)

target_label_ids = [idx for idx, name in enumerate(label_names) if name != reference_label]
steering_df = evaluate_direction_steering(
    embeddings=raw_test_embeddings,
    true_label_ids=test_label_ids,
    direction_vectors=raw_directions,
    classifier_weight=classifier_weight,
    classifier_bias=classifier_bias,
    label_names=label_names,
    target_label_ids=target_label_ids,
    alphas=[0.25, 0.5, 1.0],
)
steering_df.to_csv(analysis_dir / "steering_summary.csv", index=False)
print("Steering summary on raw final-layer test embeddings:")
display(steering_df.round(4))

In [ ]:
print("Analysis directory:", analysis_dir)
for path in sorted(analysis_dir.iterdir()):
    print(path.name)

## Optional Google Drive Sync

This sync step copies the full analysis output directory to the same Google Drive run folder used by notebook 02.

Destination:
`MyDrive/speech-emotion-directions/runs/<experiment_name>/analysis/`

In [ ]:
import shutil

if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping on local execution.')
elif drive_analysis_dir is None:
    print('Drive analysis directory is not configured.')
else:
    if drive_analysis_dir.exists():
        shutil.rmtree(drive_analysis_dir)
    drive_analysis_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(analysis_dir, drive_analysis_dir)

    print('Backed up analysis artifacts to Google Drive:')
    print(' -', drive_analysis_dir)
    print('\nDrive analysis contents:')
    for path in sorted(drive_analysis_dir.rglob('*')):
        relative = path.relative_to(drive_analysis_dir)
        if path.is_dir():
            print(f' [dir]  {relative}')
        else:
            print(f' [file] {relative}')